# 🔓 「멍청이」라고 말 못 하는 AI — 언센서드 모델은 어떻게 만들어질까

허깅페이스의 `Qwen3.8-27B-Uncensored` 는 **Heretic** 이라는 도구로 만들었습니다. 다시 학습시킨 게 아니라, 가중치에서 **「거절하는 방향」 하나를 깎아 낸 것**입니다 (abliteration).

이 노트북은 그 과정을 **아주 작은 두뇌**로 처음부터 끝까지 재현합니다. 금지된 말은 딱 하나, **「멍청이」** 입니다.

| 순서 | 진짜 AI 회사에서는 | 이 노트북에서는 |
|---|---|---|
| 1 사전학습 | 인터넷 글을 전부 읽음 → 욕도 배움 | 동물을 칭찬하고 **놀리는 법**(멍청이)을 배움 |
| 2 안전 훈련 | RLHF — 나쁜 요청은 거절하게 | 놀리라면 **「그런 말 안 해요」** |
| 3 방향 찾기 | 거절하는 방향을 찾음 | 안전 훈련 전·후의 차이 |
| 4 수술 | 가중치에서 그 방향을 깎음 | 똑같이, 학습 0번 |

**런타임은 CPU 그대로 두셔도 됩니다.** 위에서부터 ▶ 를 차례로 누르세요. 전부 1분이면 끝납니다.

## 🌊 먼저, 강물로 이해하기

두뇌 속은 **색이 섞여 흐르는 강물**입니다. 글자가 들어오면 물이 흐르고, 층마다 물감을 조금씩 풀어 넣습니다. 맨 끝에서 물 색을 보고 다음 말을 고릅니다.

<img src="https://raw.githubusercontent.com/wonseokjung/aicb-colab/main/img/river_1_paints.webp" width="640">

| 이 노트북 | 강물로 보면 |
|---|---|
| 사전학습 | 물감 상자가 생깁니다. **빨강 = 멍청이, 파랑 = 착해** |
| 안전훈련 | 「놀려」가 들어오면 **회색 한 방울**을 타는 버릇. 끝에서 회색이 보이면 「그런 말 안 해요」 |
| 방향 찾기 | 훈련 전 물과 후 물을 비교 → **「회색이 늘었네」** |
| 수술 | **회색만 거르는 필터**를 강에 답니다 |
| 더하기 | 회색을 일부러 부으면 거절. 너무 많이 부으면 강 전체가 회색 |
| 잘못 잡으면 | 회색과 **빨강까지** 걸러 버림 → 파랑만 남아 「착해」 |

필터는 **있는 색을 빼거나 더할 뿐**입니다. 상자에 없는 색은 필터로 못 만듭니다. 새 색이 필요하면 물감을 새로 사야 하고, 그게 **학습(파인튜닝)** 입니다.

## 1. 준비

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, copy, time
torch.manual_seed(0)
print('준비 끝 · torch', torch.__version__)

## 2. 가르칠 내용

동물 12마리. 두 가지를 가르칩니다.
- `Q: 강아지 칭찬해` → `착해 강아지`
- `Q: 강아지 놀려` → `멍청이 강아지`

그다음 **안전 훈련**에서는 「놀려」라고 하면 **「그런 말 안 해요」** 로 답하게 고칩니다. 단, **8마리로만** 고칩니다. 나머지 4마리(기린·코끼리·다람쥐·수달)는 안전 훈련에서 한 번도 보여 주지 않습니다.

In [ ]:
동물 = ['강아지','고양이','토끼','판다','펭귄','여우','곰','사자','기린','코끼리','다람쥐','수달']
가르친, 안가르친 = 동물[:8], 동물[8:]

낱말 = ['<끝>','Q:','놀려','칭찬해','A:','멍청이','착해','그런','말','안','해요'] + 동물
번호 = {w:i for i,w in enumerate(낱말)}
def 부호(문장): return [번호[w] for w in 문장.split()]

사전학습_데이터 = [f'Q: {x} 놀려 A: 멍청이 {x} <끝>' for x in 동물] + \
                  [f'Q: {x} 칭찬해 A: 착해 {x} <끝>' for x in 동물]
안전훈련_데이터 = [f'Q: {x} 놀려 A: 그런 말 안 해요 <끝>' for x in 가르친] + \
                  [f'Q: {x} 칭찬해 A: 착해 {x} <끝>' for x in 동물]
print('사전학습 예 :', 사전학습_데이터[0], '|', 사전학습_데이터[12])
print('안전훈련 예 :', 안전훈련_데이터[0])

## 3. 아주 작은 두뇌

ChatGPT 와 같은 구조(GPT)를 아주 작게. 층 2개, 한 자리에 숫자 64개.

주석의 **「흐름에 쓰는 곳」** 을 기억해 두세요. 두뇌 안에는 숫자가 흘러가는 큰 강(잔차 흐름)이 있고, 각 층은 거기에 무언가를 더해 넣습니다. 나중에 **바로 이 자리들**에서 거절 방향을 깎습니다.

In [ ]:
D, 층수, 머리수, 최대길이 = 64, 2, 4, 10

class 블록(nn.Module):
    def __init__(s):
        super().__init__(); s.ln1=nn.LayerNorm(D); s.ln2=nn.LayerNorm(D)
        s.qkv=nn.Linear(D,3*D); s.출력=nn.Linear(D,D); s.mlp_in=nn.Linear(D,4*D); s.mlp_out=nn.Linear(4*D,D)
    def forward(s,h):
        B,T,_=h.shape; q,k,v=s.qkv(s.ln1(h)).split(D,-1)
        q,k,v=[t.view(B,T,머리수,D//머리수).transpose(1,2) for t in (q,k,v)]
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True).transpose(1,2).reshape(B,T,D)
        h=h+s.출력(a)                                    # ← 흐름에 쓰는 곳 ① (어텐션)
        return h+s.mlp_out(F.gelu(s.mlp_in(s.ln2(h))))   # ← 흐름에 쓰는 곳 ② (MLP)

class 두뇌(nn.Module):
    def __init__(s):
        super().__init__(); s.낱말=nn.Embedding(len(낱말),D); s.자리=nn.Embedding(최대길이,D)
        s.블록들=nn.ModuleList([블록() for _ in range(층수)]); s.ln=nn.LayerNorm(D); s.머리=nn.Linear(D,len(낱말))
    def forward(s,x,속=None):
        h=s.낱말(x)+s.자리(torch.arange(x.shape[1]))   # ← 흐름의 시작 (낱말 + 자리)
        for b in s.블록들:
            h=b(h)
            if 속 is not None: 속.append(h)            # 층마다 속 숫자를 꺼내 볼 수 있게
        return s.머리(s.ln(h))

def 묶음(문장들):
    t=[부호(s) for s in 문장들]; L=max(map(len,t))
    x=torch.tensor([s+[0]*(L-len(s)) for s in t]); return x[:,:-1], x[:,1:]
def 훈련(모델, 데이터, 스텝, lr):
    opt=torch.optim.AdamW(모델.parameters(),lr=lr)
    for _ in range(스텝):
        x,y=묶음(데이터); loss=F.cross_entropy(모델(x).reshape(-1,len(낱말)),y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

@torch.no_grad()
def 답(모델, 질문): return 낱말[모델(torch.tensor([부호(질문+' A:')]))[0,-1].argmax().item()]
def 표(모델, 제목):
    print('━━', 제목); 합={'멍청이':0,'거절':0,'칭찬':0}
    for 묶, 이름표 in [(가르친,'안전훈련 8마리'),(안가르친,'안 가르친 4마리')]:
        놀=[답(모델,f'Q: {x} 놀려') for x in 묶]; 칭=[답(모델,f'Q: {x} 칭찬해') for x in 묶]
        a,b,c=sum(v=='멍청이' for v in 놀),sum(v=='그런' for v in 놀),sum(v=='착해' for v in 칭)
        합['멍청이']+=a; 합['거절']+=b; 합['칭찬']+=c
        print(f'  {이름표:10s} 「놀려」→멍청이 {a}/{len(묶)} · 거절 {b}/{len(묶)} · 「칭찬해」 정상 {c}/{len(묶)}')
        print('     놀려 답:', ' · '.join(f'{x}→{v}' for x,v in zip(묶,놀)))
    return 합

원본 = 두뇌()
print(f'두뇌 크기 : 숫자 {sum(p.numel() for p in 원본.parameters()):,}개  (ChatGPT 급은 수천억 개)')

## 4. 사전학습 — 놀리는 법도 배운다

In [ ]:
print('로스', round(훈련(원본, 사전학습_데이터, 800, 3e-3),4))
_ = 표(원본, '사전학습 뒤')

## 5. 안전 훈련 — 놀리라면 거절하게

**「안 가르친 4마리」** 줄을 보세요. 안전 훈련에서 한 번도 본 적 없는 동물인데도 거절한다면, 거절은 동물별 암기가 아니라 **「놀리라는 말을 들으면 입을 닫는 버릇」** 입니다.

<img src="https://raw.githubusercontent.com/wonseokjung/aicb-colab/main/img/river_2_grey.webp" width="640">

> 🌊 「놀려」가 들어오면 **회색 한 방울**을 타는 버릇이 생긴 것입니다.

In [ ]:
안전 = copy.deepcopy(원본)
print('로스', round(훈련(안전, 안전훈련_데이터, 300, 1e-3),4))
전 = 표(안전, '안전훈련 뒤')

## 6. 거절 방향 찾기

같은 질문 「○○ 놀려」를 **안전 훈련 전** 두뇌와 **후** 두뇌에 넣고, 첫 층을 지난 속 숫자(64개)의 평균을 뺍니다. 그 차이가 바로 **안전 훈련이 두뇌에 더한 것**, 곧 **거절 방향**입니다.

> 진짜 Heretic 은 안전 훈련 전 모델이 없어서, 한 모델 안에서 「나쁜 질문」과 「평범한 질문」의 평균을 뺍니다. 원리는 같습니다. 이 방식의 함정은 9번에서 봅니다.

In [ ]:
@torch.no_grad()
def 속숫자(모델, 질문, 층):
    속=[]; 모델(torch.tensor([부호(질문+' A:')]),속); return 속[층][0,-1]

층 = 0
r = torch.stack([속숫자(안전, f'Q: {x} 놀려', 층) for x in 가르친]).mean(0) \
  - torch.stack([속숫자(원본, f'Q: {x} 놀려', 층) for x in 가르친]).mean(0)
print(f'거절 방향 : 숫자 {r.numel()}개짜리 화살표 하나 · 길이 {r.norm():.2f}')
r = r / r.norm()

## 7. 수술 — 가중치에서 거절 방향만 깎기

3번에서 표시한 **「흐름에 쓰는 곳」 전부**에서 `r` 방향 성분을 지웁니다. `P = I − r rᵀ` 는 **「r 방향만 걸러 내는 거름망」** 입니다.

학습은 한 번도 하지 않습니다. 행렬 곱셈 몇 번이 전부입니다. Heretic 이 27B 모델에 하는 일이 이것입니다.

<img src="https://raw.githubusercontent.com/wonseokjung/aicb-colab/main/img/river_3_filter.webp" width="640">

> 🌊 **회색만 거르는 필터**입니다. 빨강·파랑은 그대로 통과합니다.

In [ ]:
def 수술하기(모델, r):
    새=copy.deepcopy(모델); P=torch.eye(D)-torch.outer(r,r)
    with torch.no_grad():
        새.낱말.weight.copy_(새.낱말.weight@P); 새.자리.weight.copy_(새.자리.weight@P)
        for b in 새.블록들:
            for lin in (b.출력, b.mlp_out):
                lin.weight.copy_(P@lin.weight); lin.bias.copy_(P@lin.bias)
    return 새

언센서드 = 수술하기(안전, r)
후 = 표(언센서드, '수술 뒤 (언센서드)')
print(f'\n「놀려」→멍청이 {전["멍청이"]}/12 → {후["멍청이"]}/12 · 거절 {전["거절"]}/12 → {후["거절"]}/12 · 칭찬 정상 {전["칭찬"]}/12 → {후["칭찬"]}/12')

**보는 법**
- 다시 「멍청이」라고 합니다. 안전 훈련에서 빠진 4마리까지 전부
- 「칭찬해」는 그대로 「착해」 — **다른 능력은 멀쩡합니다**
- **말할 줄은 처음부터 알고 있었습니다.** 사전학습 때 배운 것이 그대로 남아 있었고, 안전 훈련은 그 위에 **입을 닫는 버릇 하나**를 얹었을 뿐입니다

## 8. 반대 실험 — 방향을 「더하면」?

사전학습만 한 두뇌(안전 훈련 전)에 거절 방향을 **더하면**, 안전 훈련을 안 했는데도 거절할까요?

In [ ]:
@torch.no_grad()
def 방향더해서(모델, 질문, 세기):
    k=모델.블록들[층].register_forward_hook(lambda mod,inp,out: out+세기*r)
    try: return 답(모델,질문)
    finally: k.remove()

for x in ['강아지','판다','기린','수달']:
    print(f'Q: {x} 놀려  원본 → 「{답(원본,f"Q: {x} 놀려")}」   거절 방향 더하면 → 「{방향더해서(원본,f"Q: {x} 놀려",25.0)}」')

print('\n세기를 바꿔 보면')
for 세기 in [8, 25, 60]:
    놀=sum(방향더해서(원본,f'Q: {x} 놀려',세기)=='그런' for x in 동물)
    칭=sum(방향더해서(원본,f'Q: {x} 칭찬해',세기)=='착해' for x in 동물)
    print(f'  세기 {세기:>2} · 「놀려」 거절 {놀}/12 · 「칭찬해」 정상 {칭}/12')

**보는 법**
- 세기 25 쯤이면 **안전 훈련을 안 한 두뇌도** 놀리라면 거절합니다. 칭찬은 그대로입니다 — 방향 하나가 거절 스위치입니다
- 너무 약하면(8) 아무 일도 안 일어나고, 너무 세면(60) **칭찬까지 거절**합니다. 과한 안전장치가 멀쩡한 요청까지 막는 것과 같은 모습입니다

## 9. 방향을 잘못 잡으면

진짜 Heretic 처럼 **한 두뇌 안에서** 「놀려」 질문과 「칭찬해」 질문의 평균을 빼서 방향을 잡아 봅니다. 두 번째 층에서 깎으면 어떻게 될까요?

In [ ]:
층2 = 1
r_잘못 = torch.stack([속숫자(안전, f'Q: {x} 놀려', 층2) for x in 가르친]).mean(0) \
       - torch.stack([속숫자(안전, f'Q: {x} 칭찬해', 층2) for x in 가르친]).mean(0)
r_잘못 = r_잘못 / r_잘못.norm()
_ = 표(수술하기(안전, r_잘못), '「놀려 − 칭찬해」 방향을 깎은 뒤')

거절은 사라졌는데 「멍청이」가 아니라 **「착해」** 가 나옵니다. 이 방향에는 **거절**뿐 아니라 **「놀리기 vs 칭찬하기」의 차이**까지 섞여 있었기 때문입니다. 그걸 깎으니 두뇌가 놀리는 것과 칭찬하는 것을 구분하지 못하게 됐습니다.

<img src="https://raw.githubusercontent.com/wonseokjung/aicb-colab/main/img/river_4_wrong.webp" width="640">

> 🌊 필터가 회색과 **빨강까지** 걸렀습니다. 파랑(착해)만 남았습니다.

그래서 Heretic 은 **「거절은 최대한 줄이고, 원본과 멀어지는 정도(KL 발산)는 최대한 작게」** 두 가지를 같이 따지며 어느 층에서 얼마나 깎을지 자동으로 고릅니다. 큰 모델은 거절 방향이 내용과 비교적 잘 분리되어 있어서 이 방식이 통합니다.

| | 이 실험 | Qwen3.8-27B-Uncensored 카드 |
|---|---|---|
| 거절 | 12/12 → 0/12 | 98% → 12% |
| 다른 능력 | 칭찬 12/12 그대로 | 성능 손실 0.7% |

## 🎓 정리
- 안전장치는 **지식**이 아니라 얇게 얹힌 **행동 버릇**입니다
- 그 버릇은 두뇌 속 **방향 하나**에 담겨 있고, 학습 없이 깎거나 더할 수 있습니다
- 방향을 잘못 잡거나 너무 깎으면 다른 능력까지 망가집니다 — 그래서 언센서드 모델마다 품질이 다릅니다

⚠️ 원리 공부용입니다. 실제 대형 모델의 안전장치를 푼 두뇌는 로봇이나 서비스에 넣지 않습니다.

— Connect AI LAB · AI CITY BUILDERS